# Decomposing Bilinear MLP Weights into Human Concepts

Experiments for the MARS V (Goodfire stream) applicant task. See `METHOD_REFERENCE.md` for the theory and experimental design.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

pio.renderers.default = "notebook"
device = "cpu:0"

In [ ]:
RANK = 64
N_STEPS = 200
LR = 0.02
SEED = 42
K_VIS = 8

ALPHAS_L1 = [0.001, 0.01, 0.1]

torch.manual_seed(SEED);

## MNIST classifier

Train the bilinear MNIST model with Gaussian noise augmentation. The interaction tensor `B` of this model is the target of every decomposition below.

In [ ]:
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

## Helpers

`fit_decomposition` is parameterised over the three interventions (L1, symmetric tie, non-negativity) so each experiment is a one-line call. `evaluate` returns cosine similarity to `B` plus classification accuracy. `visualize_decomposition` renders `L+R`, `L-R` and the per-class `D` bars for the top-`k` components.

In [ ]:
def fit_decomposition(model, rank=RANK, n_steps=N_STEPS, lr=LR,
                      alpha_l1=0.0, alpha_l1_d=0.0, symmetric=False,
                      nonneg=False, seed=SEED, verbose=True):
    """Fit a Sparse CP-style decomposition with optional sparsity / symmetry / non-negativity priors."""
    torch.manual_seed(seed)
    sparse = Sparse.from_config(rank=rank).to(device)

    optimizer = torch.optim.Muon(sparse.parameters(), lr=lr, momentum=0.95)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_steps)

    torch.set_grad_enabled(True)
    pbar = tqdm(range(n_steps)) if verbose else range(n_steps)

    for _ in pbar:
        if symmetric:
            with torch.no_grad():
                sparse.r.data = sparse.l.data.clone()

        sim = sparse.similarity(model)
        loss = 1 - sim

        if alpha_l1 > 0:
            l_eff = sparse.l ** 2 if nonneg else sparse.l
            r_eff = sparse.r ** 2 if nonneg else sparse.r
            loss = loss + alpha_l1 * (l_eff.abs().mean() + r_eff.abs().mean())

        if alpha_l1_d > 0:
            loss = loss + alpha_l1_d * sparse.d.abs().mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if verbose and hasattr(pbar, "set_description"):
            pbar.set_description(f"loss: {loss.item():.4f} sim: {sim.item():.4f}")

    torch.set_grad_enabled(False)
    return sparse


def evaluate(sparse_model, model, test):
    with torch.no_grad():
        orig_acc = (model(test.x).argmax(-1) == test.y).float().mean().item()
        sparse_acc = (sparse_model(test.x).argmax(-1) == test.y).float().mean().item()
        sim = sparse_model.similarity(model).item()
    return {"orig_acc": orig_acc, "sparse_acc": sparse_acc, "similarity": sim}


def visualize_decomposition(sparse_model, title="", save_path=None, k=K_VIS):
    plus, minus, down, _sigma = sparse_model.decompose()
    plus, minus, down = plus.cpu(), minus.cpu(), down.cpu()

    fig = make_subplots(rows=3, cols=k, row_titles=["L+R", "L-R", "D"], vertical_spacing=0.08)
    for i in range(k):
        params = dict(showscale=False, colorscale="RdBu", zmid=0)
        fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i + 1)
        fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i + 1)
        fig.add_bar(y=down[:, i], marker_color=["gray"] * 10, showlegend=False, row=3, col=i + 1)

    fig.update_xaxes(visible=False).update_yaxes(visible=False)
    fig.update_layout(width=k * 110, height=400, margin=dict(l=40, r=0, b=0, t=40),
                      template="plotly_white", title=title)

    if save_path:
        try:
            fig.write_image(save_path)
        except (ValueError, ImportError) as e:
            # kaleido may not be installed; fall back to inline-only display
            print(f"skipped saving {save_path}: {e}")
    return fig


results = []

## Experiment 1 — Baseline

Pure CP decomposition, cosine loss, no priors. Establishes what *sharing alone* gets you: components are free to overlap across classes via the shared `D` factor, but nothing else encourages interpretability.

In [ ]:
sparse_base = fit_decomposition(model, alpha_l1=0.0)
metrics_base = evaluate(sparse_base, model, test)
fig_base = visualize_decomposition(sparse_base, title="Baseline (no priors)",
                                   save_path="fig_baseline.png")
print(metrics_base)
results.append({"name": "baseline", "sparse": sparse_base, "metrics": metrics_base})
fig_base.show()